In [2]:
# Importing libraries

import xarray as xr
import glob
import numpy as np
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import dask.array as da
import lightgbm as lgb
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
from lightgbm import DaskLGBMRegressor
import pandas as pd

In [3]:
parq_files = {
    'ssrd': 'flat_data_parquet/ssrd.parquet',
    'strd': 'flat_data_parquet/strd.parquet',
    'tcc':  'flat_data_parquet/tcc.parquet',
    'lsm':  'flat_data_parquet/lsm.parquet',
    'z':    'flat_data_parquet/z.parquet',
    't2m':  'flat_data_parquet/t2m.parquet',
    'sst':  'flat_data_parquet/sst.parquet',
}
time_keys = ['valid_time', 'latitude', 'longitude']
checkpoint_dir = 'flat_data_parquet/checkpoints'
part_size = "40MB"   
land_threshold = 0.5  
test_yrs = 2          

renames = {
    '2t': 't2m',
    'temperature_2m': 't2m',
    'air_temperature': 't2m',
    '2m_temperature': 't2m',
    'sea_surface_temperature': 'sst',
    'land_sea_mask': 'lsm',
    'lsm_lookup': 'lsm',
    'total_cloud_cover': 'tcc',
    'tcc_lookup': 'tcc',
    'surface_solar_radiation_downwards': 'ssrd',
    'surface_thermal_radiation_downwards': 'strd',
    'geopotential': 'z'
}

os.makedirs(checkpoint_dir, exist_ok=True)

In [4]:
# Setting up Dask cluster

cluster = LocalCluster(
    n_workers=1,           
    memory_limit='10GB',   
    processes=False,      
    silence_logs='error'
)

client = Client(cluster)
print("Dask client connected:", client)

Dask client connected: <Client: 'inproc://192.168.29.14/11736/1' processes=1 threads=16, memory=9.31 GiB>


2026-07-18 18:42:24,423 - distributed.worker - ERROR - Exception during execution of task ('hash-join-transfer-367fbf76d00071aaa11a6f4674172554', 38)
Traceback (most recent call last):
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\distributed\worker.py", line 2148, in execute
    data[dkey] = self.data[dkey]
                 ~~~~~~~~~^^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\distributed\spill.py", line 223, in __getitem__
    return super().__getitem__(key)
           ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\common.py", line 127, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\buffer.py", line 186, in __getitem__
    return self.slow_to_fast(key)
           ~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\buffer.py", line 153, in slow_to_fast
    value = self.slow[key]
            ~~~~~~~~~^

SystemError: deallocated bytearray object has exported buffers

2026-07-18 18:42:31,692 - distributed.worker - ERROR - Exception during execution of task ('hash-join-transfer-367fbf76d00071aaa11a6f4674172554', 43)
Traceback (most recent call last):
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\distributed\worker.py", line 2148, in execute
    data[dkey] = self.data[dkey]
                 ~~~~~~~~~^^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\distributed\spill.py", line 223, in __getitem__
    return super().__getitem__(key)
           ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\common.py", line 127, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\buffer.py", line 186, in __getitem__
    return self.slow_to_fast(key)
           ~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\buffer.py", line 153, in slow_to_fast
    value = self.slow[key]
            ~~~~~~~~~^

SystemError: deallocated bytearray object has exported buffers

2026-07-18 18:42:33,163 - distributed.worker - ERROR - Exception during execution of task ('hash-join-transfer-367fbf76d00071aaa11a6f4674172554', 45)
Traceback (most recent call last):
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\distributed\worker.py", line 2148, in execute
    data[dkey] = self.data[dkey]
                 ~~~~~~~~~^^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\distributed\spill.py", line 223, in __getitem__
    return super().__getitem__(key)
           ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\common.py", line 127, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\buffer.py", line 186, in __getitem__
    return self.slow_to_fast(key)
           ~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\zict\buffer.py", line 153, in slow_to_fast
    value = self.slow[key]
            ~~~~~~~~~^

In [7]:
def load_normalize(path):
    df = dd.read_parquet(path, blocksize=part_size)
    df = df.rename(columns={k: v for k, v in renames.items() if k in df.columns})
    float_cols = [c for c in df.columns if df[c].dtype == 'float64']
    if float_cols:
        df = df.astype({c: 'float32' for c in float_cols})
    return df


def merge_and_checkpoint(left_ddf, right_path, checkpoint_path):
    right_ddf = load_normalize(right_path)
    merged = left_ddf.merge(right_ddf, on=time_keys, how='inner', shuffle_method='p2p')
    merged.to_parquet(checkpoint_path, engine='pyarrow', write_index=False)
    return dd.read_parquet(checkpoint_path, blocksize=part_size)



In [9]:
names = list(parq_files.keys())

first_name = names[0]
ddf = load_normalize(parq_files[first_name])
print(f"Loaded {first_name}: {ddf.npartitions} partitions")

prev_checkpoint = None 
for i, name in enumerate(names[1:], start=1):
    checkpoint_path = f"{checkpoint_dir}/step_{i}_{name}.parquet"
    ddf = merge_and_checkpoint(ddf, parq_files[name], checkpoint_path)
    print(f"After merging {name}: {ddf.npartitions} partitions")

    if prev_checkpoint is not None and os.path.exists(prev_checkpoint):
        shutil.rmtree(prev_checkpoint)
        print(f"Deleted intermediate checkpoint: {prev_checkpoint}")
    prev_checkpoint = checkpoint_path

print(f"Final columns: {ddf.columns.tolist()}")

required = ['t2m', 'sst', 'lsm', 'tcc', 'ssrd', 'strd', 'z']
missing = [c for c in required if c not in ddf.columns]
if missing:
    raise KeyError(
        f"Missing expected columns after all merges: {missing}. "
        f"Actual columns: {ddf.columns.tolist()}. "
        f"Add the correct mapping to RENAME_MAP above."
    )


Loaded ssrd: 77 partitions


MemoryError: 

In [ ]:
# =========================================================================
# 4. Build the land/sea target using the land-sea mask
# =========================================================================
ddf['target'] = ddf['t2m'].where(ddf['lsm'] >= LAND_THRESHOLD, ddf['sst'])
ddf = ddf.dropna(subset=['target'])

TARGET = 'target'

# t2m and sst directly encode the target depending on lsm, so they must be
# dropped as features or the model trivially leaks the answer. lsm itself
# stays in as a feature — it's a legitimate predictor (coastal proximity
# affects temperature behavior on both land and sea).
drop_cols = [TARGET, 't2m', 'sst', 'valid_time', 'latitude', 'longitude']
feature_cols = [c for c in ddf.columns if c not in drop_cols]
print(f"Feature columns: {feature_cols}")


# =========================================================================
# 5. Train/test split — split by YEAR, not randomly
# =========================================================================
# A random row split would let the model see near-identical timestamps
# from the same year in both train and test, inflating apparent
# performance. Holding out whole years measures real generalization.
ddf['year'] = ddf['valid_time'].dt.year
years = sorted(ddf['year'].unique().compute().tolist())
print(f"Years available: {years}")

train_years = years[:-N_TEST_YEARS]
test_years = years[-N_TEST_YEARS:]
print(f"Train years: {train_years}")
print(f"Test years:  {test_years}")

ddf_train = ddf[ddf['year'].isin(train_years)]
ddf_test = ddf[ddf['year'].isin(test_years)]

X_train, y_train = ddf_train[feature_cols], ddf_train[TARGET]
X_test, y_test = ddf_test[feature_cols], ddf_test[TARGET]

X_train, X_test, y_train, y_test = client.persist(
    [X_train, X_test, y_train, y_test]
)


# =========================================================================
# 6. Train with the Dask-native LightGBM estimator
# =========================================================================
model = lgb_dask.DaskLGBMRegressor(
    client=client,
    objective='regression',
    metric='rmse',
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=1000,
    feature_fraction=0.9,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)],
)


# =========================================================================
# 7. Evaluate
# =========================================================================
# Only the (small) evaluation slice gets pulled into memory, not the
# full training set.
y_pred = model.predict(X_test).compute()
y_true = y_test.compute()

mae = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")


# =========================================================================
# 8. Feature importance
# =========================================================================
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.booster_.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(importance)


# =========================================================================
# 9. Save model
# =========================================================================
model.booster_.save_model('lgbm_model.txt')


# =========================================================================
# 10. Clean up
# =========================================================================
client.close()
cluster.close()

# Once you've confirmed the model trained successfully, you can delete
# the checkpoint directory to reclaim disk space:
#   import shutil; shutil.rmtree(CHECKPOINT_DIR)

In [2]:
path = "C:/Users/idasi/Documents/ideas_tih/IDEAS-TIH-Summer-2026/data/processed"
files = sorted((glob.glob(os.path.join(path, "*.nc"))))
print(f"Total files found: {len(files)}")

Total files found: 120


In [3]:
dset = xr.open_mfdataset(
    files,
    parallel=True,
    engine='netcdf4',
    data_vars='minimal',
    chunks={'valid_time': 720, 'latitude': 125, 'longitude': 121}
    )

print(dset)

print("Chunks:", dset['t2m'].chunks)
print("Type:", type(dset['t2m'].data))

c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.

<xarray.Dataset> Size: 37GB
Dimensions:     (valid_time: 87672, latitude: 125, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T2...
    expver      (valid_time) <U4 1MB dask.array<chunksize=(720,), meta=np.ndarray>
  * latitude    (latitude) float64 1kB 36.0 35.75 35.5 35.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 63.0 63.25 63.5 ... 92.5 92.75 93.0
    number      int64 8B 0
Data variables:
    ssrd        (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    strd        (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    sst         (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    tcc         (valid_time, latitude, longitude) float32 5GB 

In [5]:
dir = os.path.join(os.getcwd(), 'flat_data_parquet')

var_ssrd = dd.read_parquet(os.path.join(dir, 'ssrd.parquet'))
var_strd = dd.read_parquet(os.path.join(dir, 'strd.parquet'))
var_tcc = dd.read_parquet(os.path.join(dir, 'tcc.parquet'))
var_lsm = dd.read_parquet(os.path.join(dir, 'lsm.parquet'))
var_z = dd.read_parquet(os.path.join(dir, 'z.parquet'))
var_t2m = dd.read_parquet(os.path.join(dir, 't2m.parquet'))
var_sst = dd.read_parquet(os.path.join(dir, 'sst.parquet'))

print(f"SSRD partitions: {var_ssrd.npartitions}")
print(f"STRD partitions: {var_strd.npartitions}")
print(f"TCC partitions: {var_tcc.npartitions}")
print(f"LSM partitions: {var_lsm.npartitions}")
print(f"Z partitions: {var_z.npartitions}")
print(f"T2M partitions: {var_t2m.npartitions}")
print(f"SST partitions: {var_sst.npartitions}")

SSRD partitions: 12
STRD partitions: 26
TCC partitions: 14
LSM partitions: 10
Z partitions: 14
T2M partitions: 23
SST partitions: 9


In [6]:
spatial_keys = ['latitude', 'longitude']
time_keys = ['valid_time', 'latitude', 'longitude']

ddf = var_ssrd.merge(var_strd, on=time_keys, how='left', shuffle_method='tasks')
ddf = ddf.merge(var_tcc, on=time_keys, how='left', shuffle_method='tasks')

first_time = var_lsm['valid_time'].head(1).iloc[0]
print(f"Using timestamp: {first_time}")

lsm_lookup = dd.read_parquet(
    'flat_data_parquet/lsm.parquet',
    filters=[('valid_time', '==', first_time)]
).compute()

z_lookup = dd.read_parquet(
    'flat_data_parquet/z.parquet',
    filters=[('valid_time', '==', first_time)]
).compute()

print(f'lsm lookup size: {len(lsm_lookup):}')
print(f'z lookup size: {len(z_lookup):}')

Using timestamp: 2016-01-01 00:00:00
lsm lookup size: 15125
z lookup size: 15125


In [7]:
spatial_keys = ['latitude', 'longitude']
time_keys = ['valid_time', 'latitude', 'longitude']

drop_cols = ['number', 'expver']

def clean(df):
    return df.drop(columns=[c for c in drop_cols if c in df.columns])

var_ssrd = clean(var_ssrd)
var_strd = clean(var_strd)
var_tcc  = clean(var_tcc)
var_lsm  = clean(var_lsm)
var_z    = clean(var_z)
var_t2m  = clean(var_t2m)
var_sst  = clean(var_sst)

ddf = var_ssrd.merge(var_strd, on=time_keys, how='inner', shuffle_method='tasks')
ddf = ddf.merge(var_tcc, on=time_keys, how='inner', shuffle_method='tasks')
ddf = ddf.merge(var_lsm, on=time_keys, how='inner', shuffle_method='tasks')
ddf = ddf.merge(var_z, on=time_keys, how='inner', shuffle_method='tasks')
ddf = ddf.merge(var_t2m, on=time_keys, how='inner', shuffle_method='tasks')
ddf = ddf.merge(var_sst, on=time_keys, how='inner', shuffle_method='tasks')

print(f"Total partitions after merge: {ddf.npartitions}")
print(f"Columns: {ddf.columns.tolist()}")


Total partitions after merge: 26
Columns: ['valid_time', 'latitude', 'longitude', 'ssrd', 'strd', 'tcc', 'lsm', 'z', 't2m', 'sst']


In [1]:
land_threshold = 0.5
ddf['target'] = ddf['t2m'].where(ddf['lsm'] >= land_threshold, ddf['sst'])

ddf = ddf.dropna(subset=['target'])

output_var = 'target'

drop_cols = [output_var, 't2m', 'sst', 'valid_time', 'latitude', 'longitude']
cols = [c for c in ddf.columns if c not in drop_cols]

X = ddf[cols]
y = ddf[output_var]

ddf['year'] = ddf['valid_time'].dt.year
years = sorted(ddf['year'].unique().compute().tolist())
print(f"Years available: {years}")

test_years = 2 
train_years = years[:-test_years]
test_years = years[-test_years:]
print(f"Train years: {train_years}")
print(f"Test years:  {test_years}")

NameError: name 'ddf' is not defined

In [ ]:
ddf_train = ddf[ddf['year'].isin(train_years)]
ddf_test = ddf[ddf['year'].isin(test_years)]

X_train, y_train = ddf_train[cols], ddf_train[TARGET]
X_test, y_test = ddf_test[cols], ddf_test[TARGET]


X_train, X_test, y_train, y_test = client.persist(
    [X_train, X_test, y_train, y_test]
)

model = lgb_dask.DaskLGBMRegressor(
    client=client,
    objective='regression',
    metric='rmse',
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=1000,
    feature_fraction=0.9,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)],
)

In [ ]:

y_pred = model.predict(X_test).compute()
y_true = y_test.compute()

mae = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")

In [ ]:
importance = pd.DataFrame({
    'feature': cols,
    'importance': model.booster_.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(importance)

model.booster_.save_model('lgbm_model.txt')

In [ ]:
client.close()
cluster.close()

In [1]:
import dask.dataframe as dd
raw = dd.read_parquet('flat_data_parquet/ssrd.parquet')
print(raw.npartitions)

12
